# S-SONDO Finetuning on ESC-50

Demonstrates downstream finetuning of S-SONDO on ESC-50 (50 environmental sound classes).

**Two modes** (toggle `FREEZE_BACKBONE` below):
- **Linear probe** (`FREEZE_BACKBONE = True`): Freeze backbone, train only a linear head. Fast, standard evaluation protocol.
- **Full finetuning** (`FREEZE_BACKBONE = False`): Train the entire model end-to-end. Higher accuracy, slower.

Both use 5-fold cross-validation following ESC-50's predefined folds.

In [31]:
import os

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset, Audio
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

from ssondo import get_ssondo

# ============================================================
# Configuration — toggle this to switch between modes
# ============================================================
FREEZE_BACKBONE = True   # True = linear probe, False = full finetuning

SEED = 42
TARGET_SR = 32000
MODEL_NAME = "matpac-mobilenetv3"
EPOCHS = 10000 if FREEZE_BACKBONE else 20
LR = 1e-3 if FREEZE_BACKBONE else 1e-4
BATCH_SIZE = 64

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = "Linear Probe (frozen backbone)" if FREEZE_BACKBONE else "Full Finetuning"
print(f'Device: {DEVICE}')
print(f'Mode:   {MODE}')

Device: cpu
Mode:   Linear Probe (frozen backbone)


## 1. Load S-SONDO Model

In [32]:
model = get_ssondo(MODEL_NAME, device=DEVICE)
if FREEZE_BACKBONE:
    model.freeze_backbone()
print(f'Model: {MODEL_NAME}')
print(f'Embedding dim: {model.embedding_dim}')
print(f'Backbone frozen: {not next(model.backbone.parameters()).requires_grad}')

Model: matpac-mobilenetv3
Embedding dim: 960
Backbone frozen: True


## 2. Load ESC-50 Dataset

In [33]:
ds = load_dataset('ashraq/esc50', split='train')
ds = ds.cast_column('audio', Audio(sampling_rate=TARGET_SR))
n_classes = len(set(ds['target']))
print(f'ESC-50: {len(ds)} samples, {n_classes} classes')

for i in range(3):
    sample = ds[i]
    audio = sample['audio']
    print(f"  Sample {i}: category='{sample['category']}', duration={len(audio['array'])/audio['sampling_rate']:.1f}s")

Repo card metadata block was not found. Setting CardData to empty.


ESC-50: 2000 samples, 50 classes
  Sample 0: category='dog', duration=5.0s
  Sample 1: category='chirping_birds', duration=5.0s
  Sample 2: category='vacuum_cleaner', duration=5.0s


## 3. Prepare Audio Data

We load all waveforms into memory. For **linear probe** mode, we also pre-extract embeddings since the backbone is frozen. For **full finetuning**, embeddings are computed on-the-fly during training.

In [34]:
all_waveforms = []
all_labels = []

for sample in tqdm(ds, desc='Loading audio'):
    audio = sample['audio']
    waveform = torch.tensor(audio['array'], dtype=torch.float32)
    if waveform.ndim > 1:
        waveform = waveform.mean(dim=0)
    min_samples = TARGET_SR * 10
    if waveform.shape[0] < min_samples:
        waveform = torch.nn.functional.pad(waveform, (0, min_samples - waveform.shape[0]))
    all_waveforms.append(waveform)
    all_labels.append(sample['target'])

waveforms = torch.stack(all_waveforms)
labels = torch.tensor(all_labels, dtype=torch.long)
print(f'Waveforms: {waveforms.shape}')

if FREEZE_BACKBONE:
    print('Pre-extracting embeddings (backbone is frozen)...')
    all_emb = []
    for wav in tqdm(waveforms, desc='Extracting embeddings'):
        with torch.no_grad():
            emb = model.get_embeddings(wav.unsqueeze(0).to(DEVICE))
        all_emb.append(emb.squeeze(0).cpu())
    embeddings = torch.stack(all_emb)
    print(f'Embeddings: {embeddings.shape}')
else:
    embeddings = None
    print('Embeddings will be computed on-the-fly during training.')

Loading audio: 100%|██████████| 2000/2000 [00:06<00:00, 293.64it/s]


Waveforms: torch.Size([2000, 320000])
Pre-extracting embeddings (backbone is frozen)...


Extracting embeddings: 100%|██████████| 2000/2000 [01:37<00:00, 20.53it/s]

Embeddings: torch.Size([2000, 960])


## 4. Train & Evaluate (5-Fold Cross-Validation)

In [35]:
def train_fold(model, waveforms, embeddings, labels, train_idx, val_idx, n_classes, device):
    """Train one fold. Uses pre-extracted embeddings if frozen, else end-to-end."""
    head = nn.Linear(model.embedding_dim, n_classes).to(device)

    if FREEZE_BACKBONE:
        # Linear probe: train head on cached embeddings
        X_train = embeddings[train_idx].to(device)
        y_train = labels[train_idx].to(device)
        X_val = embeddings[val_idx].to(device)
        y_val = labels[val_idx].to(device)
        optimizer = torch.optim.Adam(head.parameters(), lr=LR)
    else:
        # Full finetuning: train backbone + head end-to-end
        model.unfreeze_backbone()
        model.train()
        optimizer = torch.optim.Adam(
            list(model.parameters()) + list(head.parameters()), lr=LR
        )

    criterion = nn.CrossEntropyLoss()
    best_acc = 0.0

    for epoch in range(EPOCHS):
        head.train()
        perm = torch.randperm(len(train_idx))

        for i in range(0, len(train_idx), BATCH_SIZE):
            batch_perm = perm[i:i + BATCH_SIZE]

            if FREEZE_BACKBONE:
                batch_emb = X_train[batch_perm]
                batch_y = y_train[batch_perm]
            else:
                batch_idx = train_idx[batch_perm]
                batch_wav = waveforms[batch_idx].to(device)
                batch_y = labels[batch_idx].to(device)
                batch_emb = model.get_embeddings(batch_wav)

            logits = head(batch_emb)
            loss = criterion(logits, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation
        head.eval()
        with torch.no_grad():
            if FREEZE_BACKBONE:
                val_emb = X_val
            else:
                model.eval()
                val_emb = model.get_embeddings(waveforms[val_idx].to(device))
                model.train()
            val_preds = head(val_emb).argmax(dim=1).cpu().numpy()
            val_labels = labels[val_idx].numpy() if FREEZE_BACKBONE else labels[val_idx].numpy()
            val_acc = accuracy_score(val_labels, val_preds)

        if val_acc > best_acc:
            best_acc = val_acc

    return best_acc

In [ ]:
fold_ids = np.array(ds['fold'])
unique_folds = sorted(set(ds['fold']))

fold_accuracies = []
for fold in unique_folds:
    val_idx = np.where(fold_ids == fold)[0]
    train_idx = np.where(fold_ids != fold)[0]

    acc = train_fold(
        model, waveforms, embeddings, labels,
        train_idx, val_idx, n_classes, DEVICE,
    )
    fold_accuracies.append(acc)
    print(f'Fold {fold}: {acc:.4f}')

mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)
print(f'\nMean Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}')

Fold 1: 0.8725
Fold 2: 0.9225
Fold 3: 0.9000
Fold 4: 0.9250


## 5. Results

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print('=' * 60)
print(f'S-SONDO FINETUNING RESULTS — {MODE}')
print('=' * 60)
print(f'Model:    {MODEL_NAME} (auto-downloaded from HF Hub)')
print(f'Dataset:  ESC-50 ({len(ds)} samples, {n_classes} classes)')
print(f'Mode:     {MODE}')
print(f'Params:   {trainable_params:,} trainable / {total_params:,} total')
print(f'Epochs:   {EPOCHS}')
print(f'LR:       {LR}')
print()
for fold, acc in zip(unique_folds, fold_accuracies):
    print(f'  Fold {fold}: {acc:.4f}')
print(f'\n  Mean Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}')
print('=' * 60)